# Load all required packages

In [3]:
import polars as pl
from PIL import Image
import polars as pl
import plotly.express as px
import numpy as np
from os.path import join as here
import optuna
import plotly.graph_objects as go

c:\Users\gcpost\AppData\Local\anaconda3\envs\peach\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Make Figures

In [9]:

df = pl.read_csv(here("archive", "LLM_HPO.csv")).join(
    pl.read_csv(here("archive","LLM_mAP.csv")),
    how="left",
    on="trial",

).rename(
    {"mean_mAP_50_95": "mAP_value"}
)

value_cols = [col for col in df.columns if col.endswith("_value")]

# Desired hyperparameter order, e.g. "hsv_h_value" -> "hsv_h"
desired_param_order = [
    col.removesuffix("_value")
    for col in value_cols
]

normalized_df = df.with_columns([
    pl.when(pl.col(col).max() == pl.col(col).min())
    .then(0.0)
    .otherwise(
        (pl.col(col) - pl.col(col).min())
        / (pl.col(col).max() - pl.col(col).min())
    )
    .alias(col)
    for col in value_cols
])

diff_df = (
    normalized_df
    .sort("trial")
    .with_columns([
        pl.col(col).diff().fill_null(0).alias(col)
        for col in value_cols
    ])
)

plot_df = diff_df.select(["trial", *value_cols]).sort("trial")



z = plot_df.select(value_cols).to_numpy()
x = [col.removesuffix("_value") for col in value_cols]
y = plot_df["trial"].to_list()

max_abs = np.nanpercentile(np.abs(z), 90)

fig = px.imshow(
    z,
    x=x,
    y=y,
    origin="lower",  # key line: makes y-axis ascend normally
    color_continuous_scale=[
        [0.0, "#2C7BB6"],
        [0.5, "#F7F7F7"],
        [1.0, "#FDAE61"],
    ],
    aspect="auto",
    zmin=-0.5,
    zmax=0.5,
    labels={
        "x": "Hyperparameters & mAP 50-95",
        "y": "Trial",
        "color": "Shift",
    },
)

fig.update_yaxes(
    tickmode="array",
    tickvals=y,
    ticktext=[str(v) for v in y],
)

fig.update_xaxes(
    showgrid=True,
    gridwidth=1,
    gridcolor="white",
    tickangle=-45,
)

fig.update_yaxes(
    showgrid=True,
    gridwidth=1,
    gridcolor="white",
  
)

fig.update_traces(
    xgap=1,
    ygap=1,
)

fig.update_layout(
    height=700,
    font=dict(
        family="Times New Roman",
        size=16,
    )
)

fig

path = here("figs", "LLM_HPO.png")
fig.write_image(path, width=800,
    height=600,
    scale=4,)

with Image.open(path) as image:
    image.save(path, dpi=(600, 600))

In [8]:
study = optuna.load_study(
    study_name="TPE",
    storage="sqlite:///archive//peach_hpo.db",
)
# Optuna gives a pandas DataFrame
trials_pd = study.trials_dataframe()

# Convert to Polars
df = pl.from_pandas(trials_pd).join(
    pl.read_csv(here("archive", "TPE_values.csv")).select(["trial", "mean_map"]).rename({"trial": "number"}),
    how="left",
    on="number",

).rename(
    {"mean_map": "params_mAP"}
)

# Trial number plus hyperparameter columns
param_cols = [col for col in df.columns if col.startswith("params_")]

# Reorder Optuna params to match LLM_HPO value_cols order
param_cols = [
    f"params_{param}"
    for param in desired_param_order
    if f"params_{param}" in param_cols
]

if "params_mAP" in df.columns and "params_mAP" not in param_cols:
    param_cols.append("params_mAP")

plot_base = (
    df
    .filter(pl.col("state") == "COMPLETE")
    .select(["number", *param_cols])
    .sort("number")
)

normalized_df = plot_base.with_columns([
    pl.when(pl.col(col).max() == pl.col(col).min())
    .then(0.0)
    .otherwise(
        (pl.col(col) - pl.col(col).min())
        / (pl.col(col).max() - pl.col(col).min())
    )
    .alias(col)
    for col in param_cols
])

diff_df = normalized_df.with_columns([
    pl.col(col).diff().fill_null(0).alias(col)
    for col in param_cols
])

z = diff_df.select(param_cols).to_numpy()
x = [col.removeprefix("params_") for col in param_cols]
y = diff_df["number"].to_list()

max_abs = np.nanpercentile(np.abs(z), 90)

fig = px.imshow(
    z,
    x=x,
    y=y,
    origin="lower",  # key line: makes y-axis ascend normally
    color_continuous_scale=[
        [0.0, "#2C7BB6"],
        [0.5, "#F7F7F7"],
        [1.0, "#FDAE61"],
    ],
    aspect="auto",
    zmin=-0.5,
    zmax=0.5,
    labels={
        "x": "Hyperparameters & mAP 50-95",
        "y": "Trial",
        "color": "Shift",
    },
)

fig.update_traces(
    xgap=1,
    ygap=1,
)

fig.update_yaxes(
    tickmode="array",
    tickvals=y,
    ticktext=[str(v) for v in y],
)

fig.update_xaxes(
    tickangle=-45,
)

fig.update_layout(
    height=700,
    font=dict(
        family="Times New Roman",
        size=16,
    )
)


path = here("figs", "TPE_HPO.png")
fig.write_image(path, width=800,
    height=600,
    scale=4,)

with Image.open(path) as image:
    image.save(path, dpi=(600, 600))

fig

In [7]:

df = pl.read_csv(here("archive", "LLM_HPO.csv")).join(
    pl.read_csv(here("archive", "LLM_mAP.csv")),
    how="left",
    on="trial",

).rename(
    {"mean_mAP_50_95": "mAP"}
)

legend_order = [
    "hsv_h",
    "hsv_s",
    "hsv_v",
    "degrees",
    "translate",
    "scale",
    "shear",
    "perspective",
    "flipud",
    "fliplr",
    "bgr",
    "mosaic",
    "mixup",
    "cutmix",
    "copy_paste",
    "close_mosaic",
    "multiple shifted",
]

marker_symbols = [
    "circle",          # hsv_h
    "square",          # hsv_s
    "diamond",         # hsv_v
    "cross",           # degrees
    "x",               # translate
    "triangle-up",     # scale
    "triangle-down",   # shear
    "triangle-left",   # perspective
    "triangle-right",  # flipud
    "pentagon",        # fliplr
    "hexagon",         # bgr
    "star",            # mosaic
    "diamond-open",    # mixup
    "square-open",     # cutmix
    "circle-open",     # copy_paste
    "octagon",         # close_mosaic
    "cross-open",      # multiple shifted
]

symbol_map = dict(zip(legend_order, marker_symbols))


# Change this to your actual score column
SCORE_COL = "mAP"   # or "mAP50-95_value", "map50_value", etc.
TRIAL_COL = "trial"

# Hyperparameter columns: adjust exclusions if needed
value_cols = [
    col for col in df.columns
    if col.endswith("_value")
]

# Normalize hyperparameter columns to 0-1 before diffing
normalized = df.with_columns([
    pl.when(pl.col(col).max() == pl.col(col).min())
    .then(0.0)
    .otherwise(
        (pl.col(col) - pl.col(col).min())
        / (pl.col(col).max() - pl.col(col).min())
    )
    .alias(col)
    for col in value_cols
])

# Absolute trial-to-trial hyperparameter shifts
diff_df = normalized.with_columns([
    pl.col(col).diff().abs().fill_null(0).alias(col)
    for col in value_cols
])

def largest_shift_label(row, cols, tolerance=0.05):
    if not cols:
        return "no hyperparameter columns found"

    changes = {col: row[col] for col in cols if row[col] is not None}

    if not changes:
        return "no shift"

    max_change = max(changes.values())

    if max_change == 0:
        return "baseline"

    close_cols = [
        col for col, value in changes.items()
        if value >= max_change * (1 - tolerance)
    ]

    if len(close_cols) > 1:
        return "multiple shifted"

    return (
        close_cols[0]
        .removeprefix("params_")
        .removesuffix("_value")
    )

shift_labels = [
    largest_shift_label(row, value_cols, tolerance=0.05)
    for row in diff_df.to_dicts()
]

plot_df = df.with_columns(
    pl.Series("largest_shift", shift_labels)
).with_columns(
    pl.col(SCORE_COL).cum_max().alias("running_best")
)

x = plot_df[TRIAL_COL].to_list()
y = plot_df[SCORE_COL].to_list()
running_best = plot_df["running_best"].to_list()
labels = plot_df["largest_shift"].to_list()

baseline = y[0]
best_idx = max(range(len(y)), key=lambda i: y[i])
best_trial = x[best_idx]
best_score = y[best_idx]

fig = go.Figure()

# Running best
fig.add_trace(go.Scatter(
    x=x,
    y=running_best,
    mode="lines",
    name="Running best",
    line=dict(color="#111827", width=2, dash="dash"),
))

# Baseline horizontal line
fig.add_hline(
    y=baseline,
    line_dash="dash",
    line_color="#94A3B8",
    annotation_text=f"First trial mAP {baseline:.5f}",
    annotation_position="right",
)

# Main line
fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode="lines",
    name="mAP per run",
    line=dict(color="#2563EB", width=3),
))

# Colored points by largest shifted hyperparameter
categories = sorted(set(labels))

for category in legend_order:
    idxs = [i for i, label in enumerate(labels) if label == category]

    if not idxs:
        continue

    fig.add_trace(go.Scatter(
        x=[x[i] for i in idxs],
        y=[y[i] for i in idxs],
        mode="markers",
        name=category,
        marker=dict(
            size=16,
            symbol=symbol_map.get(category, "circle"),
            line=dict(width=1, color="white"),
        ),
        customdata=[labels[i] for i in idxs],
        hovertemplate=(
            "Trial: %{x}<br>"
            f"{SCORE_COL}: " + "%{y:.5f}<br>"
            "Largest shift: %{customdata}<extra></extra>"
        ),
    ))





# Highlight best point
fig.add_trace(go.Scatter(
    x=[best_trial],
    y=[best_score],
    mode="markers",
    name="Best",
    marker=dict(
        size=18,
        color="white",
        line=dict(color="#111827", width=3),
    ),
    showlegend=False,
))

fig.add_annotation(
    x=best_trial,
    y=best_score,
    text=f"Best: trial {best_trial}, {SCORE_COL} {best_score:.6f}",
    showarrow=False,
    yshift=22,
    font=dict(size=16, color="#111827"),
)

fig.update_layout(
    xaxis_title="Trial",
    yaxis_title="Mean mAP 50-95",
    template="plotly_white",
    width=1200,
    height=700,
    legend_title="Legend",
    font=dict(
        family="Times New Roman",
        size=16,
    )
)

fig.update_xaxes(
    tickmode="array",
    tickvals=x,
    showgrid=True,
)

fig.update_yaxes(showgrid=True)

path = here("figs", "LLM_HPO_line.png")

fig.write_image(path, width=1200,
    height=700,
    scale=4,)

with Image.open(path) as image:
    image.save(path, dpi=(600, 600))

fig

In [6]:
study = optuna.load_study(
    study_name="TPE",
    storage="sqlite:///archive//peach_hpo.db",
)

df = pl.from_pandas(study.trials_dataframe())

desired_param_order = [
    "hsv_h",
    "hsv_s",
    "hsv_v",
    "degrees",
    "translate",
    "scale",
    "shear",
    "perspective",
    "flipud",
    "fliplr",
    "bgr",
    "mosaic",
    "mixup",
    "cutmix",
    "copy_paste",
    "close_mosaic",
]

param_cols = [
    f"params_{param}"
    for param in desired_param_order
    if f"params_{param}" in df.columns
]

plot_base = (
    df
    .filter(pl.col("state") == "COMPLETE")
    .select(["number", "value", *param_cols])
    .sort("number")
)

normalized = plot_base.with_columns([
    pl.when(pl.col(col).max() == pl.col(col).min())
    .then(0.0)
    .otherwise(
        (pl.col(col) - pl.col(col).min())
        / (pl.col(col).max() - pl.col(col).min())
    )
    .alias(col)
    for col in param_cols
])

diff_df = normalized.with_columns([
    pl.col(col).diff().abs().fill_null(0).alias(col)
    for col in param_cols
])

def largest_shift_label(row, cols, tolerance=0.05):
    changes = {col: row[col] for col in cols if row[col] is not None}

    if not changes:
        return "no shift"

    max_change = max(changes.values())

    if max_change == 0:
        return "baseline"

    close_cols = [
        col for col, value in changes.items()
        if value >= max_change * (1 - tolerance)
    ]

    if len(close_cols) > 1:
        return "multiple shifted"

    return close_cols[0].removeprefix("params_")

labels = [
    largest_shift_label(row, param_cols, tolerance=0.05)
    for row in diff_df.to_dicts()
]

plot_df = plot_base.with_columns(
    pl.Series("largest_shift", labels)
).with_columns(
    pl.col("value").cum_max().alias("running_best")
)

x = plot_df["number"].to_list()
y = plot_df["value"].to_list()
running_best = plot_df["running_best"].to_list()
labels = plot_df["largest_shift"].to_list()

baseline = y[0]
best_idx = max(range(len(y)), key=lambda i: y[i])
best_trial = x[best_idx]
best_score = y[best_idx]

fig = go.Figure()

# Running best
fig.add_trace(go.Scatter(
    x=x,
    y=running_best,
    mode="lines",
    name="Running best",
    line=dict(color="#111827", width=2, dash="dash"),
))

# Baseline horizontal line
fig.add_hline(
    y=baseline,
    line_dash="dash",
    line_color="#94A3B8",
    annotation_text=f"First trial mAP {baseline:.5f}",
    annotation_position="right",
    annotation_font=dict(size=16, color="#111827"),
)

# Main line
fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode="lines",
    name="mAP per run",
    line=dict(color="#2563EB", width=3),
))

# Colored points by largest shifted hyperparameter
categories = sorted(set(labels))

for category in legend_order:
    idxs = [i for i, label in enumerate(labels) if label == category]

    if not idxs:
        continue

    fig.add_trace(go.Scatter(
        x=[x[i] for i in idxs],
        y=[y[i] for i in idxs],
        mode="markers",
        name=category,
        marker=dict(
            size=16,
            symbol=symbol_map.get(category, "circle"),
            line=dict(width=1, color="white"),
        ),
        customdata=[labels[i] for i in idxs],
        hovertemplate=(
            "Trial: %{x}<br>"
            f"{SCORE_COL}: " + "%{y:.5f}<br>"
            "Largest shift: %{customdata}<extra></extra>"
        ),
    ))


# Highlight best point
fig.add_trace(go.Scatter(
    x=[best_trial],
    y=[best_score],
    mode="markers",
    name="Best",
    marker=dict(
        size=18,
        color="white",
        line=dict(color="#111827", width=3),
    ),
    showlegend=False,
))

fig.add_annotation(
    x=best_trial,
    y=best_score,
    text=f"Best: trial {best_trial}, {SCORE_COL} {best_score:.6f}",
    showarrow=False,
    yshift=22,
    font=dict(size=16, color="#111827"),
)

fig.update_layout(
    xaxis_title="Trial",
    yaxis_title="Mean mAP 50-95",
    template="plotly_white",
    width=1200,
    height=700,
    legend_title="Legend",
    legend=dict(
        y=0.70,
        yanchor="top",
        x=1.0,
        xanchor="left",
    ),
    font=dict(
        family="Times New Roman",
        size=16,
    )
)

fig.update_xaxes(
    tickmode="array",
    tickvals=x,
    showgrid=True,
)

fig.update_yaxes(showgrid=True)

path = here("figs", "TPE_HPO_line.png")

fig.write_image(path, width=1200,
    height=700,
    scale=4,)

with Image.open(path) as image:
    image.save(path, dpi=(600, 600))

fig